# Bonus 1 — Advanced Data Cleaning & Preprocessing

The baseline pipeline in Module 1 loads PDFs and splits them naively.  
Real insurance documents have quirks that hurt retrieval quality:

- **Headers / footers** on every page (document title, page numbers, legal disclaimers)
- **Tables** rendered as garbled text when extracted linearly
- **Multilingual content** — a single DKV document may mix FR, NL, and EN
- **Near-duplicate chunks** — repeated boilerplate across chapters
- **Very short chunks** — section titles, page numbers extracted as standalone chunks

**Exercises:** Steps 1, 2, 4, and 5 have TODOs to implement.  
Steps 3 and 6 are walkthroughs — run them as-is and observe the output.


In [ ]:
import os
import re
import warnings
warnings.filterwarnings('ignore')

import pdfplumber
import pandas as pd
import numpy as np
from pathlib import Path
from langdetect import detect, LangDetectException
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from dotenv import load_dotenv

load_dotenv('../.env')

DATA_DIR = Path('../data/sample_dkv')
pdf_files = sorted(DATA_DIR.glob('*.pdf'))
print(f'Found {len(pdf_files)} PDFs: {[p.name for p in pdf_files]}')


---
## Step 1 — Extract text with pdfplumber (structure-aware)

`pdfplumber` can extract tables as structured objects and gives per-page bounding boxes,  
making it easier to strip headers/footers by position.


In [ ]:
def extract_pages(pdf_path: Path) -> list[dict]:
    """Extract pages with text and tables, filtering headers/footers by Y position."""
    pages = []
    with pdfplumber.open(pdf_path) as pdf:
        page_height = pdf.pages[0].height if pdf.pages else 800

        # TODO: Define header_cutoff and footer_cutoff as fractions of page_height.
        # The header band is the top 8% of the page; the footer band is the bottom 8%.
        header_cutoff = ...  # page_height * ?
        footer_cutoff = ...  # page_height * ?

        for page_num, page in enumerate(pdf.pages, 1):
            # TODO: Extract words using page.extract_words() and keep only those
            # whose 'top' coordinate falls between header_cutoff and footer_cutoff.
            # Each word dict has keys 'text' and 'top'. Cast 'top' to float.
            words = ...
            body_text = ' '.join(words).strip()

            # TODO: Extract tables using page.extract_tables().
            # For each table, convert rows to pipe-separated strings.
            # Skip empty rows (any(row) == False). Append the joined rows
            # as a single '\n'.join(rows) string if rows is non-empty.
            tables_md = []
            # your table extraction here

            pages.append({
                'source': pdf_path.name,
                'page': page_num,
                'text': body_text,
                'tables': tables_md,
            })
    return pages

# Extract all PDFs
all_pages = []
for pdf in pdf_files:
    pages = extract_pages(pdf)
    all_pages.extend(pages)
    print(f'{pdf.name}: {len(pages)} pages, {sum(len(p["tables"]) for p in pages)} tables')

print(f'\nTotal: {len(all_pages)} pages extracted')


---
## Step 2 — Text normalisation

Insurance PDFs often contain OCR artifacts, ligatures, and inconsistent whitespace.


In [ ]:
_LIGATURES = {
    '\ufb01': 'fi', '\ufb02': 'fl', '\ufb03': 'ffi', '\ufb04': 'ffl',
    '\u2019': "'", '\u2018': "'", '\u201c': '"', '\u201d': '"',
    '\u2013': '-', '\u2014': '-', '\u00a0': ' ',
}

def normalise(text: str) -> str:
    # TODO: Replace each key in _LIGATURES with its value.

    # TODO: Collapse runs of spaces/tabs to a single space (keep newlines intact).
    # Hint: re.sub(r'[ \t]+', ' ', text)

    # TODO: Remove lines that contain only a page number (1–3 digits, optional whitespace).
    # Hint: re.sub(r'(?m)^\s*\d{1,3}\s*$', '', text)

    # TODO: Collapse 3 or more consecutive newlines down to 2.

    return text.strip()

# Apply normalisation
for p in all_pages:
    p['text'] = normalise(p['text'])

sample = all_pages[0]
print(f'Page 1 of {sample["source"]} ({len(sample["text"])} chars after normalisation):')
print(sample['text'][:400])


---
## Step 3 — Language detection

DKV Belgium documents may be FR, NL, or EN.  
Knowing the language per chunk lets you route queries to the right embedding model  
or filter by language in a multilingual deployment.


In [ ]:
def detect_language(text: str) -> str:
    """Return ISO 639-1 code or 'unknown' for short/ambiguous text."""
    if len(text.split()) < 10:
        return 'unknown'
    try:
        return detect(text)
    except LangDetectException:
        return 'unknown'

for p in all_pages:
    p['language'] = detect_language(p['text'])

lang_counts = pd.Series([p['language'] for p in all_pages]).value_counts()
print('Language distribution across pages:')
print(lang_counts.to_string())


---
## Step 4 — Quality filtering

Short chunks (< 50 words) are usually section titles, captions, or extraction noise —  
they add no retrieval value and dilute embedding space.


In [ ]:
# TODO: Choose a minimum word count and filter all_pages.
# Short pages (section titles, captions, stray numbers) hurt retrieval quality.
# Hint: len(p['text'].split()) gives the word count for page p.

MIN_WORDS = ...  # try 50

before = len(all_pages)
all_pages = ...  # keep only pages with >= MIN_WORDS words
print(f'Quality filter: {before} → {len(all_pages)} pages ({before - len(all_pages)} dropped)')

word_counts = [len(p['text'].split()) for p in all_pages]
print(f'Word count: min={min(word_counts)}, median={int(np.median(word_counts))}, max={max(word_counts)}')


---
## Step 5 — Near-duplicate detection

Boilerplate sections (legal disclaimers, contact info) appear on multiple pages.  
Duplicate chunks waste embedding space and skew retrieval toward repeated content.

Strategy: embed all chunks, then flag pairs with cosine similarity > 0.97.


In [ ]:
embed_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

texts = [p['text'][:512] for p in all_pages]  # truncate for speed
print(f'Embedding {len(texts)} pages...')
embeddings = embed_model.encode(texts, show_progress_bar=True, batch_size=32)

SIM_THRESHOLD = 0.97

# TODO:
# 1. Compute the full pairwise cosine_similarity(embeddings) matrix.
# 2. Zero out the diagonal so a page isn't flagged as its own duplicate.
#    Hint: np.fill_diagonal(matrix, 0)
# 3. Build a 'duplicates' set:
#    For each page i (skip if already in duplicates):
#      Find all j where sim_matrix[i, j] > SIM_THRESHOLD and j > i.
#      Add those j to duplicates (keep the first occurrence, discard later ones).
# 4. Build all_pages_deduped by keeping only pages whose index is NOT in duplicates.
# 5. If any duplicates were found, print an example pair.

duplicates = set()
# your implementation here

all_pages_deduped = ...
print(f'Near-duplicates found: {len(duplicates)}')
print(f'After deduplication: {len(all_pages)} → {len(all_pages_deduped)} pages')


---
## Step 6 — Compare retrieval quality: baseline vs cleaned

Index both the raw and cleaned chunks into separate ChromaDB collections,  
then compare retrieval precision on a few sample queries.


In [ ]:
import chromadb
import sys
sys.path.insert(0, '..')
from src.embedder import embed_texts

chroma = chromadb.EphemeralClient()

def index_pages(pages: list[dict], collection_name: str):
    try:
        chroma.delete_collection(collection_name)
    except Exception:
        pass
    col = chroma.create_collection(collection_name, metadata={'hnsw:space': 'cosine'})
    texts = [p['text'] for p in pages]
    embeddings = embed_model.encode(texts, batch_size=32, show_progress_bar=False).tolist()
    col.upsert(
        ids=[f'{p["source"]}::p{p["page"]}' for p in pages],
        documents=texts,
        embeddings=embeddings,
        metadatas=[{'source': p['source'], 'page': p['page'], 'language': p.get('language', 'unknown')} for p in pages],
    )
    return col

print('Indexing cleaned pages...')
col_clean = index_pages(all_pages_deduped, 'workshop_rag_clean')
print(f'Clean collection: {col_clean.count()} docs')

# Compare a query
test_query = 'What is covered under DKV hospitalisation insurance?'
q_emb = embed_model.encode([test_query]).tolist()

for name, col in [('workshop_rag', chroma.get_or_create_collection('workshop_rag', metadata={'hnsw:space': 'cosine'})),
                  ('workshop_rag_clean', col_clean)]:
    results = col.query(query_embeddings=q_emb, n_results=3, include=['documents', 'distances'])
    print(f'\n{name} top-3 results:')
    for doc, dist in zip(results['documents'][0], results['distances'][0]):
        print(f'  [{1-dist:.2f}] {doc[:100]}...')


---
## Summary

| Step | Effect |
|------|--------|
| Header/footer removal | Eliminates noise from positional extraction |
| Table extraction | Tables become searchable text rows |
| Normalisation | Fixes encoding/ligature artifacts |
| Quality filter | Removes short non-informative chunks |
| Deduplication | Prevents boilerplate dominating retrieval |

**Key takeaway:** Clean input → better embeddings → more precise retrieval.  
The evaluation metrics from Module 4 are the quantitative proof.
